# &#x1f5c4; Getting to know the cache!

This page provides a quick guide to the cache handled by `DiskCheckpointer` in `mgnipy.V2.mixins`.

Every proxy / resource-specific MGnifier *query* (i.e., of specific params) has its own deterministic cache key / subdirectory in a containing `config.cache_dir`. 

The cache subdirs are derived from the resource name plus the query parameters via `hashlib.sha256` 

In that subdir there will be a manifest file and a json file per request num. 

### An example of what it looks like: 

```bash
the_main_cache_dir/
├── 3fddd8853bdd0204eeaeda6c5b9b42b48c8a25ca4f034132d94eb1f93e01ac48/
│   ├── manifest.json
│   ├── page_1.json
│   ├── page_2.json
│   └── ...
├── hash_for_some_other_query/
│   ├── manifest.json
│   ├── page_1.json
│   └── ...
└── ...
```

### How writing to cache is handled

- Response items for a given page/request num are stored on disk in correspondible `page_<n>.json`
- The responses are cached after every made request 
- If the response already exists in the cache (i.e., page_1.json exists) then the response is derived from the cache rather than making another request.
- A `manifest.json` file stores the query details such as the given params and resource being searched. (more info on the manifest [below](#the-manifest))


### How loading from cache is handled

- At every proxy / resource-specific MGnifier instantiation any existing records in cache are attempted to be loaded into the instance, such as to `MGnifier().results`
- If you don't want these to be loaded from the cache, then clear it before instantiating the query


### Below are guides for:
1. Configuring the cache (#to-cache-or-not-to-cache)
1. &#x1f5fa; [Finding where the cache is stored](#locating-cache)
2. &#x1f4d1; [What type of files are in the cache](#inspecting-cache)
2. &#x1f9fd; How do I inspect or clear the cache?

---

## Locating Cache

The outtermost containing directory is configured with the `mgnipy.MGnipyConfig`. More information can be found on the [config setup page](TODO)

For your given mgnipy / mgnifier instance this cache directory path can be found using `.cache_dir`

### &#x1f5c3; Setting and finding the main cache directory

The cache directory is configured via `mgnipy.MGnipyConfig` of by passing 

More information can be found on the [config setup page](TODO)

In [9]:
[a for a in pa.iterdir() if a.is_dir()]

[PosixPath('/Users/phanthanourak/Library/Caches/mgnipy/f51d311fd384bcde35a1463c21202307bf27c9fa18cd05ecb652510ea4e295f0'),
 PosixPath('/Users/phanthanourak/Library/Caches/mgnipy/505de64d0b31a57c3290486c85289b70babc8d4157ccb5f43a91f5109424494b'),
 PosixPath('/Users/phanthanourak/Library/Caches/mgnipy/1ea63a6efb9652f69b09e9cecf801e17f053cf189f567423586eb67084a3ea13')]

In [1]:
# if using MGnipyConfig directly
from mgnipy import MGnipyConfig
config = MGnipyConfig(cache_dir="./tmp")

you can find the `cache_dir` already from the MGnipy delegator / client:

In [2]:
from mgnipy import MGnipy 
MG = MGnipy(
    config=config,
    # or
    # cache_dir="./tmp"
    )

MG.cache_dir

PosixPath('tmp')

and also from the resource-specific MGnifiers aka proxies if wanted:

In [ ]:
# init samples proxy 
samples = MG.samples
print("general cache dir:", samples.config.cache_dir)

general cache dir: tmp


### &#x1f4c1; Finding the sub-cache corresponding to a query

The cache subdirs or cache keys within `config.cache_dir` are derived from the resource name plus the query parameters via `hashlib.sha256` 

Here is how to find the full path

In [13]:
# option 1: .cache_dir 
print("Planned cache directory based on params and resource:\n", samples.cache_dir)

Planned cache directory based on params and resource:
 tmp/3fddd8853bdd0204eeaeda6c5b9b42b48c8a25ca4f034132d94eb1f93e01ac48


The cache directory path is also included in the string representation of the proxy instance :) 

In [ ]:
# option 2: __str__
print(samples)

MGnifier instance for resource: samples
I.e., mgnipy.V2.proxies.Samples
----------------------------------------
Base URL: https://www.ebi.ac.uk/
Parameters: {}
Example request URL: https://www.ebi.ac.uk/metagenomics/api/v2/samples?page=1
Endpoint module: mgnipy.emgapi_v2_client.api.samples.list_mgnify_samples
Is list endpoint (returns paginated results): True
Cache directory: tmp/3fddd8853bdd0204eeaeda6c5b9b42b48c8a25ca4f034132d94eb1f93e01ac48



## Inspecting Cache

### &#x1f4dc; The Manifest

1. the MGnify API `resource` the requests were being made to e.g., biomes, biome, studies, analyses
2. The query `params` e.g. accession, page_size, search  
3. `count` of the total items/records for the entire query
    - for list endpoints this would be the total number of listed items across all paginated responses
    - for detail endpoints the count would be 1 (as a detail corresponds to a single accession/id) or 0 if no match 
4. while `total_pages` corresponds to the total number of request urls to obtain all `count` num of items
    - for list endpoints the total_pages or number of requests is dependent on `count` / page_size param (i.e., max number of items to return for a request, default is `page_size=25` items)
    - for detail endpoints the total_pages would simply be 1 or 0 again. 

### &#x1f5d2; The pages

- Each page_#.json corresponds to a given request_num
- if list endpoint then the items are listed in the json 
- if detail endpoint then just one item in page_1.json